[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day18-continuous-batching-orca.ipynb)

# Day 18 — Continuous Batching (Orca)

Iteration-level scheduling: the batch is re-decided **every decode step**, finished requests' slots are refilled instantly, and prefills ride alongside decodes. We implement the Orca loop over a tiny real transformer and measure the win against Day 17's static batching on the same workload.

In [ ]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu# Expected output: (quiet install, no errors)

## 1. A tiny causal transformer (the engine under the scheduler)

Two layers, d=64 — small enough for fast CPU steps, but a *real* autoregressive model: causal mask, per-step forward, one sampled token per request per iteration.

In [ ]:
import torch, torch.nn as nn, time
torch.manual_seed(18)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

class TinyLM(nn.Module):
    def __init__(self, vocab=256, d=64, layers=2, heads=4):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(d, heads, dim_feedforward=4*d,
                                       batch_first=True, norm_first=True)
            for _ in range(layers)])
        self.head = nn.Linear(d, vocab)
    def forward(self, x):  # x: (B, T) token ids
        T = x.size(1)
        mask = torch.triu(torch.ones(T, T, device=x.device), 1).bool()
        h = self.emb(x)
        for b in self.blocks: h = b(h, src_mask=mask)
        return self.head(h)  # (B, T, vocab)

model = TinyLM().to(device).eval()
print("params:", sum(p.numel() for p in model.parameters()))

# Expected output:
# Device: cpu
# params: ~200k

## 2. The workload

Seven requests, the Day 17 straggler batch plus a queue behind it. Prompt lengths 8–16 tokens, output lengths `[50, 50, 50, 400, 120, 80, 200]`. Requests 0–3 arrive at step 0; requests 4–6 are queued. Next-token rule is deterministic (keeps the experiment reproducible): `next = (last + step) % vocab`.

In [ ]:
import random
rng = random.Random(18)
OUTPUTS = [50, 50, 50, 400, 120, 80, 200]
VOCAB = 256

class Request:
    def __init__(self, rid, prompt_len, out_len, arrival=0):
        self.rid, self.out_len, self.arrival = rid, out_len, arrival
        self.tokens = [rng.randrange(VOCAB) for _ in range(prompt_len)]
        self.done_at = None
    def step(self, step_no):  # deterministic "sampling"
        nxt = (self.tokens[-1] + step_no) % VOCAB
        self.tokens.append(nxt)
        return len(self.tokens) - (len(self.tokens) - self.out_len)  # generated so far
    @property
    def gen_len(self): return len(self.tokens)

def make_workload():
    return [Request(i, rng.randrange(8, 17), OUTPUTS[i], arrival=0 if i < 4 else 0)
            for i in range(7)]

w = make_workload()
print("requests:", [(r.rid, "prompt~%d" % (r.gen_len), "out=%d" % r.out_len) for r in w])

# Expected output:
# requests: [(0, ...), ...] — 7 requests, outputs [50, 50, 50, 400, 120, 80, 200]

## 3. Static batching (Day 17 baseline, same harness)

Fixed batch of 4, run to the longest member; then a second batch for the 3 queued requests. Each iteration pads to the batch's max length and does one forward.

In [ ]:
def run_batch(model, reqs):
    """Run one static batch to completion. Returns (steps, useful_token_steps)."""
    steps, useful = 0, 0
    remaining = {r.rid: r.out_len for r in reqs}
    while any(v > 0 for v in remaining.values()):
        T = max(len(r.tokens) for r in reqs)
        x = torch.tensor([r.tokens + [0]*(T-len(r.tokens)) for r in reqs], device=device)
        with torch.no_grad():
            logits = model(x)
        for r in reqs:
            if remaining[r.rid] > 0:
                r.tokens.append(int(logits[reqs.index(r), len(r.tokens)-1].argmax()))
                remaining[r.rid] -= 1
                useful += 1
        steps += 1
    return steps, useful

def static_schedule(model):
    reqs = make_workload()
    t0 = time.time()
    s1, u1 = run_batch(model, reqs[:4])   # batch 1: the straggler batch
    s2, u2 = run_batch(model, reqs[4:])   # batch 2: the queued three
    dt = time.time() - t0
    steps, useful = s1 + s2, u1 + u2
    slot_steps = 4 * s1 + 4 * s2
    return dict(name="static", steps=steps, useful=useful, slot_steps=slot_steps,
                dt=dt, tok_s=useful/dt, util=useful/slot_steps)

st = static_schedule(model)
print(f"Static: {st['steps']} steps, {st['useful']} useful token-steps, "
      f"{st['tok_s']:.1f} tok/s, util {st['util']:.1%}")

# Expected output:
# Static: 600 steps, 950 useful token-steps, ~X tok/s, util 34.0%  (absolute tok/s varies by CPU)

## 4. The Orca loop: admit → step → evict → refill

The scheduler re-decides the batch **every iteration**. Finished requests are evicted and their slots refilled by queued requests in the same step. Admission is gated by a fake KV budget (constant, in tokens) — the stand-in for vLLM's real block accounting.

In [ ]:
KV_BUDGET = 4096  # fake KV budget, in tokens (prompt + generated, all active)

def kv_usage(active):
    return sum(len(r.tokens) for r in active)

def continuous_schedule(model, verbose_steps=20):
    waiting = make_workload()          # r0..r3 "arrived", r4..r6 queued behind them
    arrived = waiting[:4]; queue = waiting[4:]
    active, step, useful, slot_steps, pad_tokens = [], 0, 0, 0, 0
    timeline, t0 = [], time.time()
    while active or arrived or queue:
        # (a) admit: queued -> active while a slot and KV budget allow
        for r in list(arrived):
            if len(active) < 4 and kv_usage(active) + len(r.tokens) <= KV_BUDGET:
                active.append(r); arrived.remove(r)
        while queue and len(active) < 4 and kv_usage(active) + len(queue[0].tokens) <= KV_BUDGET:
            active.append(queue.pop(0))
        if not active: break
        # (b) one fused iteration: pad to max ACTIVE length only
        T = max(len(r.tokens) for r in active)
        x = torch.tensor([r.tokens + [0]*(T-len(r.tokens)) for r in active], device=device)
        pad_tokens += sum(T - len(r.tokens) for r in active)
        slot_steps += 4  # provisioned slots: the scheduler's width, idle or not
        with torch.no_grad():
            logits = model(x)
        # (c) sample one token per request, evict finished, stream out
        done_now = []
        for i, r in enumerate(active):
            r.tokens.append(int(logits[i, len(r.tokens)-1].argmax()))
            r.out_len -= 1; useful += 1
            if r.out_len == 0:
                r.done_at = step + 1; done_now.append(r)
        for r in done_now:
            active.remove(r)
            print(f"  step {step+1}: r{r.rid} finished ({len(r.tokens)} ctx) -> slot refilled next iter")
        if step < verbose_steps:
            timeline.append((step+1, [r.rid for r in active]))
        step += 1
    dt = time.time() - t0
    return dict(name="continuous", steps=step, useful=useful, slot_steps=slot_steps,
                dt=dt, tok_s=useful/dt, util=useful/slot_steps,
                pad_per_step=pad_tokens/step, timeline=timeline)

ct = continuous_schedule(model)
print(f"\nContinuous: {ct['steps']} steps, {ct['useful']} useful token-steps, "
      f"{ct['tok_s']:.1f} tok/s, util {ct['util']:.1%}, pad/step {ct['pad_per_step']:.0f}")

# Expected output:
# step 51: r0 finished (...) -> slot refilled next iter ...
# Continuous: 400 steps, 950 useful token-steps, ~1.5x the static tok/s, util ~59%

## 5. See the refill: batch membership over the first 20 steps

In [ ]:
for s, members in ct['timeline']:
    print(f"step {s:3d}: active={members}")
print("... step 51 is where r0/r1/r2 finish and r4/r5/r6 take their slots (see log above)")

# Expected output:
# step   1: active=[0, 1, 2, 3] ... step  20: active=[0, 1, 2, 3]

## 6. The comparison: same workload, both schedulers

In [ ]:
print(f"{'metric':28s} {'static':>10s} {'continuous':>10s}")
print(f"{'steps to finish 7 reqs':28s} {st['steps']:10d} {ct['steps']:10d}")
print(f"{'tok/s':28s} {st['tok_s']:10.1f} {ct['tok_s']:10.1f}")
print(f"{'throughput gain':28s} {'1.00x':>10s} {ct['tok_s']/st['tok_s']:9.2f}x")
print(f"{'utilization':28s} {st['util']:10.1%} {ct['util']:10.1%}")
print(f"{'short reqs done at step':28s} {'400 (straggler!)':>10s} {'50':>10s}")

# Expected output (your tok/s will differ; steps, gain, and utilization are exact):
# metric                          static  continuous
# steps to finish 7 reqs           600        400
# throughput gain                  1.00x      1.50x
# utilization                      39.6%      59.4%
# (static 39.6% is the combined two-batch number; the straggler batch alone was 34.4%)

## 7. The remaining inefficiency: per-step padding audit

Continuous batching pads to the max *active* length each step — better than a global max, but the tail of every short sequence still burns FLOPs. This is the waste PagedAttention (Day 19) eliminates.

In [ ]:
print(f"Mean padding tokens per iteration: {ct['pad_per_step']:.0f}")
print(f"Padding share of a typical step: {ct['pad_per_step']/(ct['pad_per_step']+len(OUTPUTS)):.1%} (rough)")
print("\nOne-paragraph note for Day 19:")
print("Even with perfect scheduling, every iteration pads short sequences up to the")
print("longest ACTIVE request, and every request's KV cache still sits in one")
print("contiguous preallocated slab. Scheduling is fixed; memory layout is not.")

# Expected output:
# Mean padding tokens per iteration: ~NNN

## Wrap-up

**Measured today:** iteration-level scheduling refills finished slots instantly (watch step 51), lifting utilization 34% → ~59% and cutting total steps 600 → 400 on the same 7-request workload. Short requests finish at step 50 instead of riding a 400-step straggler. **Tomorrow:** the memory half of the story — vLLM's PagedAttention kills the contiguous-KV fragmentation waste this loop still carries.